# Notebook 01 — Train Networks (Laptop / CPU)

Trains three ReLU MLP classifiers and saves them to `models/`.

| Model | Dataset | Architecture | Est. Time (CPU) |
|---|---|---|---|
| `small_mnist` | MNIST | 784→64→64→10 | ~8 min |
| `medium_mnist` | MNIST | 784→256→256→256→10 | ~20 min |
| `large_cifar` | CIFAR-10 | 3072→512→512→512→512→10 | ~35 min |

**Total: ~60 minutes on a modern laptop CPU.**

You can run each section independently — each saves its own `.pt` file immediately after training.

**All networks use only ReLU** — no BatchNorm, no Dropout.  
This is required to preserve piecewise-linear structure for activation-region analysis.

**Outputs saved to:**
```
models/small_mnist.pt
models/medium_mnist.pt
models/large_cifar.pt
results/training_summary.json
```

## 0 — Install dependencies

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'scipy', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)

import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()} (not needed — CPU is fine)')

PyTorch : 2.12.0+cpu
CUDA    : False (not needed — CPU is fine)


## 1 — Imports & configuration

In [2]:
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from pathlib import Path
from tqdm import tqdm

# ── path setup ────────────────────────────────────────────────────────────────
REPO_ROOT = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.network_definitions import SmallMLP, MediumMLP, LargeMLP, save_model, load_model
from utils.metrics import compute_accuracy, save_results

# ── directories ───────────────────────────────────────────────────────────────
MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
for d in [MODELS_DIR, RESULTS_DIR, DATA_DIR]:
    d.mkdir(exist_ok=True)

# ── CPU settings ──────────────────────────────────────────────────────────────
DEVICE      = 'cpu'
NUM_WORKERS = 0          # 0 = main process only (safe on all OS)
SEED        = 42

torch.manual_seed(SEED)
np.random.seed(SEED)

# use all available cores
n_cores = os.cpu_count() or 2
torch.set_num_threads(n_cores)
print(f'CPU threads : {n_cores}')

CPU threads : 8


## 2 — Data loaders

In [3]:
# ── MNIST ─────────────────────────────────────────────────────────────────────
mnist_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_train = datasets.MNIST(DATA_DIR, train=True,  download=True, transform=mnist_tf)
mnist_test  = datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf)
mnist_train_loader = DataLoader(mnist_train, batch_size=128, shuffle=True,  num_workers=NUM_WORKERS)
mnist_test_loader  = DataLoader(mnist_test,  batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
print(f'MNIST    train={len(mnist_train):,}  test={len(mnist_test):,}')

# ── CIFAR-10 ──────────────────────────────────────────────────────────────────
cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914, 0.4822, 0.4465),
                         (0.2023, 0.1994, 0.2010)),
])
cifar_train = datasets.CIFAR10(DATA_DIR, train=True,  download=True, transform=cifar_tf)
cifar_test  = datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_tf)
cifar_train_loader = DataLoader(cifar_train, batch_size=128, shuffle=True,  num_workers=NUM_WORKERS)
cifar_test_loader  = DataLoader(cifar_test,  batch_size=256, shuffle=False, num_workers=NUM_WORKERS)
print(f'CIFAR-10 train={len(cifar_train):,}  test={len(cifar_test):,}')

100%|██████████████████████████████████████████████████████████████████████████████| 9.91M/9.91M [00:18<00:00, 538kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 28.9k/28.9k [00:00<00:00, 106kB/s]
100%|██████████████████████████████████████████████████████████████████████████████| 1.65M/1.65M [00:03<00:00, 484kB/s]
100%|█████████████████████████████████████████████████████████████████████████████████████| 4.54k/4.54k [00:00<?, ?B/s]


MNIST    train=60,000  test=10,000


100%|████████████████████████████████████████████████████████████████████████████████| 170M/170M [03:38<00:00, 779kB/s]


CIFAR-10 train=50,000  test=10,000


## 3 — Training function

In [4]:
def train_model(model, train_loader, test_loader,
                epochs=20, lr=1e-3, weight_decay=1e-4, model_name=''):
    """
    CPU training loop with tqdm progress bar.
    Prints loss/accuracy live so you can stop early if satisfied.
    """
    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    history   = {'train_loss': [], 'train_acc': [], 'test_acc': []}
    best_acc  = 0.0
    t0        = time.time()

    bar = tqdm(range(1, epochs + 1), desc=model_name, unit='ep')
    for epoch in bar:
        model.train()
        total_loss = correct = total = 0
        for X, y in train_loader:
            X = X.view(X.size(0), -1)
            optimizer.zero_grad()
            out  = model(X)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * y.size(0)
            correct    += (out.argmax(1) == y).sum().item()
            total      += y.size(0)
        scheduler.step()

        tr_loss = total_loss / total
        tr_acc  = correct   / total
        te_acc  = compute_accuracy(model, test_loader, DEVICE)
        best_acc = max(best_acc, te_acc)

        history['train_loss'].append(round(tr_loss, 4))
        history['train_acc'].append(round(tr_acc,   4))
        history['test_acc'].append(round(te_acc,   4))

        bar.set_postfix(loss=f'{tr_loss:.3f}',
                        train=f'{tr_acc*100:.1f}%',
                        test=f'{te_acc*100:.1f}%',
                        best=f'{best_acc*100:.1f}%',
                        t=f'{time.time()-t0:.0f}s')

    model.eval()
    print(f'  Best test acc: {best_acc*100:.2f}%  |  '
          f'Total time: {time.time()-t0:.0f}s')
    return history

## 4 — Train SmallMLP on MNIST
**Expected:** ~97% accuracy in ~8 min

In [5]:
small_model = SmallMLP(input_dim=784, output_dim=10)
print(small_model)
print(f'ReLU neurons: {small_model.n_relu_neurons()}  |  '
      f'Params: {sum(p.numel() for p in small_model.parameters()):,}\n')

small_history = train_model(
    small_model, mnist_train_loader, mnist_test_loader,
    epochs=20, lr=1e-3, model_name='SmallMLP-MNIST',
)

save_model(small_model, str(MODELS_DIR / 'small_mnist.pt'), metadata={
    'dataset': 'MNIST', 'architecture': '784-64-64-10',
    'best_test_acc'  : max(small_history['test_acc']),
    'n_relu_neurons' : small_model.n_relu_neurons(),
    'n_params'       : sum(p.numel() for p in small_model.parameters()),
})

SmallMLP(input=784, hidden=[64, 64], output=10, params=55,050)
ReLU neurons: 128  |  Params: 55,050



SmallMLP-MNIST: 100%|█████████| 20/20 [05:56<00:00, 17.85s/ep, best=98.0%, loss=0.008, t=357s, test=98.0%, train=99.9%]

  Best test acc: 97.98%  |  Total time: 357s
  Saved → D:\concolic_exploration\models\small_mnist.pt


## 5 — Train MediumMLP on MNIST
**Expected:** ~98% accuracy in ~20 min

In [6]:
medium_model = MediumMLP(input_dim=784, output_dim=10)
print(medium_model)
print(f'ReLU neurons: {medium_model.n_relu_neurons()}  |  '
      f'Params: {sum(p.numel() for p in medium_model.parameters()):,}\n')

medium_history = train_model(
    medium_model, mnist_train_loader, mnist_test_loader,
    epochs=25, lr=1e-3, model_name='MediumMLP-MNIST',
)

save_model(medium_model, str(MODELS_DIR / 'medium_mnist.pt'), metadata={
    'dataset': 'MNIST', 'architecture': '784-256-256-256-10',
    'best_test_acc'  : max(medium_history['test_acc']),
    'n_relu_neurons' : medium_model.n_relu_neurons(),
    'n_params'       : sum(p.numel() for p in medium_model.parameters()),
})

MediumMLP(input=784, hidden=[256, 256, 256], output=10, params=335,114)
ReLU neurons: 768  |  Params: 335,114



MediumMLP-MNIST: 100%|███████| 25/25 [09:30<00:00, 22.82s/ep, best=98.5%, loss=0.001, t=571s, test=98.4%, train=100.0%]

  Best test acc: 98.53%  |  Total time: 571s
  Saved → D:\concolic_exploration\models\medium_mnist.pt


## 6 — Train LargeMLP on CIFAR-10
**Expected:** ~52% accuracy in ~35 min  
> Flat MLP on CIFAR-10 won't reach CNN accuracy — that's expected and fine.  
> We only need ≥50% for robustness analysis to be meaningful.  
> The key value is showing how concolic runtime scales with network size.

In [7]:
large_model = LargeMLP(input_dim=3072, output_dim=10)
print(large_model)
print(f'ReLU neurons: {large_model.n_relu_neurons()}  |  '
      f'Params: {sum(p.numel() for p in large_model.parameters()):,}\n')

large_history = train_model(
    large_model, cifar_train_loader, cifar_test_loader,
    epochs=30, lr=5e-4, model_name='LargeMLP-CIFAR10',
)

save_model(large_model, str(MODELS_DIR / 'large_cifar.pt'), metadata={
    'dataset': 'CIFAR-10', 'architecture': '3072-512-512-512-512-10',
    'best_test_acc'  : max(large_history['test_acc']),
    'n_relu_neurons' : large_model.n_relu_neurons(),
    'n_params'       : sum(p.numel() for p in large_model.parameters()),
})

LargeMLP(input=3072, hidden=[512, 512, 512, 512], output=10, params=2,366,474)
ReLU neurons: 2048  |  Params: 2,366,474



LargeMLP-CIFAR10: 100%|██████| 30/30 [13:49<00:00, 27.64s/ep, best=55.9%, loss=0.001, t=829s, test=55.8%, train=100.0%]

  Best test acc: 55.93%  |  Total time: 829s
  Saved → D:\concolic_exploration\models\large_cifar.pt


## 7 — Save summary & verify all models

In [8]:
summary = {
    'small_mnist'  : {'history': small_history,
                      'best_test_acc': max(small_history['test_acc']),
                      'architecture' : '784-64-64-10',
                      'n_relu_neurons': small_model.n_relu_neurons(),
                      'n_params'     : sum(p.numel() for p in small_model.parameters())},
    'medium_mnist' : {'history': medium_history,
                      'best_test_acc': max(medium_history['test_acc']),
                      'architecture' : '784-256-256-256-10',
                      'n_relu_neurons': medium_model.n_relu_neurons(),
                      'n_params'     : sum(p.numel() for p in medium_model.parameters())},
    'large_cifar'  : {'history': large_history,
                      'best_test_acc': max(large_history['test_acc']),
                      'architecture' : '3072-512-512-512-512-10',
                      'n_relu_neurons': large_model.n_relu_neurons(),
                      'n_params'     : sum(p.numel() for p in large_model.parameters())},
}
save_results(summary, str(RESULTS_DIR / 'training_summary.json'))

# ── reload & verify ───────────────────────────────────────────────────────────
print('\nVerifying saved models...')
for name, path in [('small_mnist',  MODELS_DIR / 'small_mnist.pt'),
                   ('medium_mnist', MODELS_DIR / 'medium_mnist.pt'),
                   ('large_cifar',  MODELS_DIR / 'large_cifar.pt')]:
    m   = load_model(str(path))
    out = m(torch.randn(1, m.input_dim))
    assert out.shape == (1, 10)
    print(f'  {name:<20} shape={list(out.shape)}  ✓')

# ── final table ───────────────────────────────────────────────────────────────
print()
print('═'*65)
print('  TRAINING COMPLETE')
print('═'*65)
print(f'  {"Model":<20} {"Architecture":<26} {"Best Acc":>9} {"ReLU n":>7}')
print('─'*65)
for name, info in summary.items():
    print(f"  {name:<20} {info['architecture']:<26}"
          f" {info['best_test_acc']*100:>8.2f}%"
          f" {info['n_relu_neurons']:>7}")
print('═'*65)
print('\n  Next step → run 02_baselines_fgsm_pgd.ipynb')

  Saved results → D:\concolic_exploration\results\training_summary.json

Verifying saved models...
  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}
  small_mnist          shape=[1, 10]  ✓
  Loaded ← D:\concolic_exploration\models\medium_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-256-256-256-10', 'best_test_acc': 0.9853, 'n_relu_neurons': 768, 'n_params': 335114}
  medium_mnist         shape=[1, 10]  ✓
  Loaded ← D:\concolic_exploration\models\large_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'architecture': '3072-512-512-512-512-10', 'best_test_acc': 0.5593, 'n_relu_neurons': 2048, 'n_params': 2366474}
  large_cifar          shape=[1, 10]  ✓

═════════════════════════════════════════════════════════════════
  TRAINING COMPLETE
═════════════════════════════════════════════════════════════════
  Model                Archite